Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import math
import os
import gc
from pathlib import Path
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Custom packages
from tools.filter import FilterDF as fdf
from tools.benchmarks import ParetoAnalysis as pa
from tools.benchmarks import AccuracyCalculation as ac
from tools.integrity_fixes import DataFixer as fix, DataExporter as exporter
from tools.coverage_functions import plot_time_series
from tools.labeling_functions import fully_relabel_and_consolidate, rename_items, rename_items_by_modifications, plot_dish_time_series

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Read data from parquet files

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
    %store static_data_merged
    
# Data already exists
else:
    static_data = static_data_merged.copy()

# 

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"data/2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"data/2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
        
    %store sales_data_merged
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

# 

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
%store -r before_after_details_true
if 'before_after_details_true' not in locals():
    before_after_details_true = pd.read_csv('data/before_after_details_true.csv', index_col='location_id')
    %store before_after_details_true

# Timezones
%store -r timezones
if 'timezones' not in locals():
    timezones = pd.read_csv('data/timezones.csv', index_col='location_id')['timezone'].to_dict()
    for loc_id, df in sales_and_menu_data.items():
        df.index = df.index.tz_convert(timezones[loc_id])
        sales_and_menu_data[loc_id] = df
    %store timezones

%store -r restaurants_by_4m_coverage
if 'restaurants_by_4m_coverage' not in locals():
    restaurants_by_4m_coverage = pd.read_csv('data/2_palate_data_parquet_cleaned/restaurants_by_4m_coverage.csv')['location_id'].tolist()
    %store restaurants_by_4m_coverage

loc_id = 'C0BE4NDSW26QN'
df_uncleaned = sales_and_menu_data[loc_id]

locations = list(sales_and_menu_data.keys())
for other_loc_id in locations:
    if other_loc_id != loc_id:
        del sales_and_menu_data[other_loc_id]
        del sales_data_merged[other_loc_id]
gc.collect()

In [ ]:
df_uncleaned['item_quantity'].sum()

In [ ]:
df_uncleaned.query('item_name.str.contains("Beyond") or item_modifications.str.contains("Beyond")')['item_name'].value_counts()

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.query('item_name.str.contains("Beyond")')['item_quantity'].sum()

In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Beyond")')['item_quantity'].sum()

In [ ]:
plot_time_series('C0BE4NDSW26QN', 
                 df_uncleaned, 
                 before_after_details_true, 
                 freq='D', 
                 subset=True)
plt.show()

In [ ]:
plot_time_series('C0BE4NDSW26QN', 
                 df_uncleaned, 
                 before_after_details_true, 
                 freq='7D', 
                 subset=False)
plt.show()

In [ ]:
plot_time_series('C0BE4NDSW26QN', 
                 df_uncleaned.query('item_name.str.contains("Beyond")'), 
                 before_after_details_true, 
                 freq='7D', 
                 subset=False)
plt.show()

In [ ]:
plot_time_series('C0BE4NDSW26QN', 
                 df_uncleaned.query('item_name.str.contains("Beyond")'), 
                 before_after_details_true, 
                 freq='D', 
                 subset=True)
plt.show()

Time Differences

In [ ]:
sales_and_menu_data[loc_id].index

In [ ]:
time_differences_details[loc_id]

# Start

Important note, this restaurant has "Unchicken" as a modification

In [ ]:
print(static_data_merged.keys())
print(sales_and_menu_data.keys())

In [ ]:
# Important note, this restaurant has "Unchicken" as a modification
df = df.assign(item_modifications = lambda df: df['item_modifications'].str.replace("Un'Chicken|Un'chicken", 'Unchicken', regex=True))
loc_id = 'C0BE4NDSW26QN'
df = sales_and_menu_data[loc_id]

# Type the entries with modifications here:

In [ ]:
# Original item name, animal-based ingredient, new name for item with animal-based ingredient
modification_name_changes = [
    ('Veggie Sandwich', 'Bacon', 'Veggie Sandwich with Bacon'),
    ('Veggie Sandwich', 'No Cheese', 'Vegan Veggie Sandwich'),
    ('Veggie Sandwich', 'Vegan', 'Vegan Veggie Sandwich'),
    ('Veggie Sandwich', 'No Chz', 'Vegan Veggie Sandwich'),
    
    ('Beyond Burger', 'Bacon', 'Beyond Burger with Bacon'),
    ('Beyond Burger', 'Vegan', 'Vegan Beyond Burger'),
    ('Beyond Burger', 'No Cheese', 'Vegan Beyond Burger'),
    ('Beyond Burger', 'No Chz', 'Vegan Beyond Burger'),
    
    ('Apple Salad', r'(?i)^(?=.*(no\s+(?:goat\s+)?cheese))(?=.*chicken).*', 'Apple Salad with Chicken'),
    ('Apple Salad', 'Chicken', 'Apple Salad with Chicken'),
    ('Apple Salad', 'No Cheese', 'Vegan Apple Salad'),
    ('Apple Salad', 'No Chz', 'Vegan Apple Salad'),
    ('Apple Salad', 'Bacon', 'Apple Salad wth Bacon'),
    
    ('Side Slaw', 'Bacon', 'Bacon Side'), # is side its own category?
    
    ('Impossible Burg', 'Bacon', 'Impossible Burg with Bacon'),
    ('Impossible Burg', r'(?i)^(?=.*no\s+mayo)(?=.*no\s+cheese).*', 'Vegan Impossible Burg'),


    ('Noble Soba Salad', 'No Chicken', 'Vegan Soba Salad'),

    ('Winter Grain Bowl', 'Vegan', 'Vegan Winter Grain Bowl'),
    ('Winter Grain Bowl', 'No Bacon Add Chicken', 'Winter Grain Bowl with Chicken'),
    ('Winter Grain Bowl', 'No Bacon', 'Vegetarian Winter Grain Bowl'),

    ('Chard', 'Vegan', 'Vegan Chard'),

    ('Summer Grain Bowl', 'Vegan', 'Vegan Summer Grain Bowl'),
    ('Summer Grain Bowl', 'No Chicken', 'Vegetarian Summer Grain Bowl'),

    ('Harvest Salad', 'Chicken', 'Harvest Salad with Chicken'),
    ('Harvest Salad', 'No Cheese', 'Vegan Harvest Salad'),

    ('Asparagus Salad', 'No Cheese', 'Vegan Asparagus Salad'),
    ('Asparagus Salad', 'Chicken', 'Asparagus Salad with Chicken'),

    ('Autumnal Grain Bowl', 'Chicken', 'Autumnal Grain Bowl with Chicken'),
    
    ('Togarashi Grain Bowl', 'Chicken', 'Togarashi Grain Bowl with Chicken'),

    ('Grain Bowl', 'Chicken', 'Grain Bowl with Chicken'),
    ('Grain Bowl', 'Vegan', 'Vegan Grain Bowl'),
    ('Watermelon Salad', 'Chicken', 'Watermelon Salad with Chicken'), # 2 people say 'no cheese' but I think this is a mistake and it's vegan by default (I looked it up and also watermelon and cheese is not a natural combination)

    ('Fried Cauliflower', 'No Feta', 'Vegan Fried Cauliflower'),
    ('Fried Cauliflower', 'Vegan', 'Vegan Fried Cauliflower'),
    
    ('Fall Apple Salad', 'Chicken', 'Fall Apple Salad with Chicken'),
    ('Fall Apple Salad', 'No Chz', 'Vegan Fall Apple Salad'),

    ('Winter Spinach Salad', 'Chicken', 'Winter Spinach Salad with Chicken'),

    ('Melon Salad', 'Chicken', 'Melon Salad with Chicken'),
    ('Melon Salad', 'No Cheese Dairy Allergy', 'Vegan Melon Salad'),

    ('Spring Salad', 'Chicken', 'Spring Salad with Meat'),
    ('Spring Salad', 'Burger', 'Spring Salad with Meat'),
    ('Spring Salad', 'No Cheese', 'Vegan Spring Salad'),
    ('Spring Salad', 'No Chz', 'Vegan Spring Salad'),
    ('Spring Salad', 'Vegan', 'Vegan Spring Salad'),

    ('Spring Market Salad', 'Chicken', 'Spring Market Salad with Chicken'),
    ('Spring Market Salad', 'No Cheese', 'Vegan Spring Market Salad'),

    ('Buffalo Cauliflower', 'No Cheese', 'Vegan Buffalo Cauliflower'),
    ('Buffalo Cauliflower', 'No Parm', 'Vegan Buffalo Cauliflower'),
    
    ('Wheatberry Salad', 'Chicken', 'Wheatberry Salad with Chicken'),
    ('Wheatberry Salad', 'Vegan', 'Vegan Wheatberry Salad'),
    ('Cauliflower App', 'Vegan', 'Vegan Cauliflower App'),
    ('Squash Salad', 'Chicken', 'Squash Salad with Chicken'),
    ('Summer Bean Salad', 'Chicken', 'Summer Bean Salad with Chicken'),
    ('Summer Bean Salad', 'Vegan', 'Vegan Summer Bean Salad'),
    
    ('Spring Bean Salad', 'No Chicken', 'Spring Bean Salad'), # some people say no chicken but there's already no chicken
    ('Spring Bean Salad', 'Chicken', 'Spring Bean Salad with Chicken'), 
    
    ('Beet And Butternut Salad', 'No Chicken', 'Beet And Butternut Salad'), # some people say no chicken but there's already no chicken
    ('Beet And Butternut Salad', 'Chicken', 'Beet And Butternut Salad with Chicken'),

    ('Peach Salad', 'Chicken', 'Peach Salad with Chicken'),
    ('Peach Salad', 'Wings', 'Peach Salad with Chicken'),

    ('Brussels Sprouts', 'No Chicken', 'Brussels Sprouts'), # some people say no chicken but there's already no chicken
    ('Brussels Sprouts', 'Chicken', 'Brussels Sprouts with Chicken'),

    ('Arugula Salad', 'No Chicken', 'Arugula Salad'), # some people say no chicken but there's already no chicken
    ('Arugula Salad', 'Chicken', 'Arugula Salad with Chicken'),

    ('Ratatouille Sandwich', 'No Cream Cheese', 'Vegan Ratatouille Sandwich'),

    ('Berry Salad', 'Chicken', 'Berry Salad with Chicken'),
    ('Berry Salad', 'No Cheese', 'Vegan Berry Salad'),

    ('Berry Panzanella Salad', 'Chicken', 'Berry Panzanella Salad with Chicken'),
    ('Berry Panzanella Salad', 'No Cheese', 'Vegan Berry Panzanella Salad'),
    ('Berry Panzanella Salad', 'No Dairy', 'Vegan Berry Panzanella Salad'),

    ('Beet Salad', 'Chicken', 'Beet Salad with Chicken'),
    ('Beet Salad', 'Vegan', 'Vegan Beet Salad'),
    ('Beet Salad', 'No Dairy', 'Vegan Beet Salad'),

    ('Strawberry And Rhubarb Salad', 'Chicken', 'Strawberry And Rhubarb Salad with Chicken'),
    ('Strawberry And Rhubarb Salad', 'No Dairy', 'Vegan Strawberry And Rhubarb Salad') ,
    
    ('Tomato N Peach', 'Chicken', 'Tomato N Peach with Chicken'),

    ('Grain Salad', 'Chicken', 'Grain Salad with Chicken'),

    ('Apple And Rhubarb Salad', 'Chicken', 'Apple And Rhubarb Salad with Chicken'),

    ('Summer Salad', 'Chicken', 'Summer Salad with Chicken'),

    ('Special - Mushroom Salad', 'Chicken', 'Mushroom Salad with Chicken'),

    ('Special Cauliflower', 'No Feta', 'Vegan Special Cauliflower'),

    ('Pizza Spring Veggie', 'Bacon', 'Pizza Spring Veggie with Bacon'),

    ('Brussel Salad', 'No Bacon', 'Brussel Salad without Bacon'),

    ('Peachsalad', 'Chicken', 'Peachsalad with chicken'),
    ('Peachsalad', 'Vegan', 'Vegan Peachsalad'),

    ('Everything Bagel Salad', 'No Salmon', 'Vegetarian Everything Bagel Salad'),

    ('Special Corn Chowda', 'No Bacon', 'Vegetarian Special Corn Chowda'),

    ('Pizza - Acadian Pie', 'No Meat', 'Vegetarian Pizza - Acadian Pie'),

    ('Pizza - Bahn Mi', 'No Pork', 'Vegetarian Pizza - Bahn Mi'),

    ("Pizza - I'M A Pepper", "Pepperoni", "Pepperoni Pizza"),

    ('Special Salad', 'Chicken', 'Special Salad with Chicken'),
]


# List the items here

In [ ]:
meat = ['Veggie Sandwich with Bacon',
        'Beyond Burger with Bacon',
        'Apple Salad with Chicken',
        'Apple Salad wth Bacon',
        'Impossible Burg with Bacon',
        'Winter Grain Bowl',
        'Winter Grain Bowl with Chicken',
        'Noble Soba Salad',
        'Chard',
        'Summer Grain Bowl',
        'Harvest Salad with Chicken',
        'Panzanella Salad',
        'Autumnal Grain Bowl with Chicken',
        'Togarashi Grain Bowl with Chicken',
        'Grain Bowl with Chicken',
        'Watermelon Salad with Chicken',
        'Fall Apple Salad with Chicken',
        'Winter Spinach Salad with Chicken',
        'Melon Salad with Chicken',
        'Spring Salad with Meat',
        'Spring Market Salad with Chicken',
        'Wheatberry Salad with Chicken',
        'Squash Salad with Chicken',
        'Summer Bean Salad with Chicken',
        'Spring Bean Salad with Chicken',
        'Beet And Butternut Salad with Chicken',
        'Peach Salad with Chicken',
        'Brussels Sprouts with Chicken',
        'Arugula Salad with Chicken',
        'Berry Salad with Chicken',
        'Berry Panzanella Salad with Chicken',
        'Beet Salad with Chicken',
        'Strawberry And Rhubarb Salad with Chicken',
        'Tomato N Peach with Chicken',
        'Grain Salad with Chicken',
        'Summer Salad with Chicken',
        'Stuffed Peppers', # https://www.instagram.com/cleveland.chocolate/p/CFiuB9TpZdj/ I think they have meat by default
        'Mushroom Salad with Chicken',
        'Pizza Spring Veggie with Bacon',
        'Brussel Salad', # has bacon by default
        'Naan Mi', # has pork: https://www.facebook.com/noblebeastbrewingco/posts/the-naan-mi-returns-to-the-menu-green-tea-and-5-spice-pork-tenderloin-wild-mushr/3588387247891530/
        'Cubano De Juan', # has pork https://www.instagram.com/noble_beast_brewing/p/B_QAfOdJO9p/
        'Mousetrap', # https://www.facebook.com/noblebeastbrewingco/posts/mousetrap-charcuterie-board-loaded-with-ohio-meats-and-cheeses-headwaters-tomme-/1718889981507942/
        'The Carina', # https://www.yelp.com/biz/noble-beast-brewing-cleveland-4?start=40 # has pork
        'Sasha',  # looks like a sandwich based on mods and most things here have meat
        'Italian Focaccia',
        'Three Little Pigs', 
        'Miso Pumpkin Melt', # think it has pork
        'Italian Wrap',
        'Cubano De Juano', # https://www.instagram.com/noble_beast_brewing/p/B_QAfOdJO9p/
        'The Club', # also a drink with this name at some point but this is a sandwich with turkey based on mods
        'Pizza Popcorn', # review says zested up with pepperoni
        'Hoagie Hoagie',
        'Brussee Sprouts', # has bacon and aoili
        'Peachsalad with chicken',
        'Baguette Sandwich',
        'Sasha Treat Of The Day',
        'Beer Burger',
        'Nye Prix Fixe Dinner',
        'Everything Bagel Salad', # has salmon
        'Special-Bagley Bagel', # this is just a guess but I bet it has salmon
        'Special Khao-Soi', # khao soi typically has chicken
        'Special - Massaman Curry', # just a guess but I assume this has chicken
        'Special - Rachel', # just a guess, it's a sandwich and most sandwiches here have meat
        'Dumps',  # dumplings, just a guess, but this is a meat-heavy place
        'Special - Burrito',
        'Special - Loaded',
        'Special - Spann Burger',
        'Special - Trojan Burger',
        'Texan Beast', # can't find any details about this but I assume it has ham/pork/whatever]
        'Special Pierogies', # probably have beef and pork in them https://www.instagram.com/noble_beast_brewing/p/CVVubTBJ414/
        'Special - Ni√±O Polacko', # https://www.instagram.com/noble_beast_brewing/p/CWd-htMJ3SX/
        'Special - Hoagie',
        'Bob Dog', # am assuming this is a kind of hot dog
        'Dumplings',
        'Special - Gas Station Bbq', # not sure about this one but the mods includes 'to go' and I guess that's food
        'Uncle Tony', # assuming meat because it's a sandwich and has an italian name
        'Special- Pierogies', # https://www.instagram.com/noble_beast_brewing/p/CVVubTBJ414/
        'Special - Melt',
        'Special- ‚Äúplt‚Äù', # I think it's supposed to be a burger or a sandwich
        "Auttuno D'Italia", # not 100% on what this is but I think it's a sandwich ('no cheese' is a mod) so I bet it has meat
        'Special - Po‚Äô Boi',
        'Sloppi Poppi',
        'Special - Nicolo Picolo', # no specific knowledge but I think it's a sandwich and they tend to have meat here
        'Special - World Cup Special', # I think it's a burger
        'Pizza Margherita',
        'Deejay‚Äôs Burgers',
        'Special - Spartan Burger',
        'The Noble Boy', # https://pbs.twimg.com/media/DGe0UdyW0AAd59O?format=jpg&name=large
        'Special - Brat', 
        'Special - Berbere Burger',
        'Special - Bob‚Äôs Banger', 
        'Hot Rachael', # Rachel is like a rueben
        '3 Little Pigs', # pork dish 
        'Special - Terroir Dog', # hot dog I think
        'Special - California Burrito',  # typically has carne asada
        'Special - Rochelle Rochelle', # most sandwiches have meat here
        'The Mitch',
        'Special - Gumbo', # https://www.instagram.com/noble_beast_brewing/p/B7ELReupIfN/
        'Bauer Outage',
        'Brat Special',
        'Frankie Lindog',
        'Special Corn Chowda',
        'Pizza - Wonton Disregard',
        'Special - 2020 Dumpster Fire',
        'Special - J-Rogis',
        'Pizza - Catalonian Spring', # I bet this has ham or equivalent
        'Special - Swinton‚Äôs Pride', # most sandwiches here have meat
        'Stuffed Pepper', 
        'Special - Hoagie Mouth', 
        'Cuban',
        'Pizza - "Foghorn Leghorn"', # https://www.instagram.com/noble_beast_brewing/p/CPq2gfip7DC/?img_index=1
        'Pizza Lamb',
        'Dumpling Soup', # am assuming here
        'Special - German Boy', # am assuming here b/c sandwich
        'Tacos',
        'Perogies',
        'The Noble Feast',
        'Pizza Dads Day',
        'Barley Swine',
        'Italian', 
        'Pizza - Acadian Pie'
        'Pizza - Bahn Mi',
        'Terrine',
        'Burgers',
        'Special: Pootie Call', # think it's a burger or a sandwich
        'Pizza - French Onion',
        'Pizza - Pierogie', 
        "Pepperoni Pizza",
        'Black N Bleu', # burger
        'Carolina', # some kind of sandwich
        'Pizza - Shepherds Pie', # assuming meat casserole on a pizza]
        'Open Face',
        'Special - Paprigosh', # I am guessing this has chicken but no specific reason to think that
        'Special Salad with Chicken',
        'Pizza - Nawlins', # i bet this has shrimp
        'Houes Burger', # typo 
        'Pickle Rick',
        'Sweet Italian',
        'Special Berbere Cheesesteak',
        'Special Stromboli',
        'Pizza - The Van Helsing', # just assuming that special pizzas have meat
        'Pizza Bahn Mi', 
        'Pizza In German',
        'Special - Cheeky Beast',
        'Special - Country Wiener',
        'Special - South Mouth',
        'Katsu Special',
        'Pizza - Fall Puy',
        'Special - Bahn Mi',
        'Llama Balls',
        'Lu Empanada',
        'Sliders',
        'Special - Johnny F‚Äôing Tsunami',
        'Special- Rakel',
        
        
       ]
        
vegetarian = ['Veggie Sandwich', 
              'Beyond Burger',
             'Apple Salad',
              'Impossible Burg',
              'Street Corn',
              'Vegetarian Summer Grain Bowl',
              'Vegetarian Winter Grain Bowl',
              'Harvest Salad',
              'Asparagus Salad',
              'Grain Bowl', 
              'Veggie Tray', # many cheeses https://www.facebook.com/photo.php?fbid=1478888645508078&id=1007683595961921&set=a.1327881637275447
              'Fried Cauliflower', # has feta
              'Melon Salad', # has feta http://facebook.com/noblebeastbrewingco/photos/melon-salad-and-grilled-elote-mexican-street-corn-melon-salad-market-melons-jica/2560185387378393/?_rdr
              'Spring Market Salad', # has cheese
              'Buffalo Cauliflower',
              'Wheatberry Salad', # dressing must not be vegan based on modifications requested
              'Cauliflower App',  # has mayo
              'Squash Salad',
              'Summer Bean Salad',
              'Peach Salad',
              'Ratatouille Sandwich',
              'Berry Salad',
              'Berry Panzanella Salad', 
              'Beet Salad',
              'Strawberry And Rhubarb Salad', 
              'Tomato N Peach',
              'Summer Salad', 
              'Potato Pizza',
              'Pizza Mushroom', 
              'Special Cauliflower',
              'Veggie Nachos',
              'Pizza Buffalo Cauli',
              'The Dalmatian',
              'Curry Salad Sammie', # I think ~everything this place has meat or cheese unless it's specifically vegan
              'Peachsalad', 
              'Pizza Spring Veggie',
              'Curry Salad Sandwich',
              'Vegetarian Everything Bagel Salad',
              'Special - Soup', # no mention of meat but mention of sour cream. this might have meat, IDK
              'Chilaquiles', # again might have meat but no evidence of it in substitutions
              'Special - ‚Äúit‚Äôs Corn!‚Äù', # some kind of corn dish
              'Popcorn Mix', # probably has butter
              'Fry Pie!!',
              'Special - Ribollita',
              'Pasta', # not a lot to go on here but am assuming it's got parmesan but that it would say if it had meat in the name
              'Pizza - Corn On The Pie', # just assuming that a corn pizza is veg but not vegan
              'Anny Sides',
              'Vegetarian Special Corn Chowda',
              'Cle Chocolate Co. Bar', # this might be vegan but I'm not sure
              'Special - Mezze', # these typically have a yogurt sauce 
              'Special - Not Your Moms Tomato Soup', # I am guessing this has cream
              'Harvest Cake',
              'Noble Mezze',
              'Vegetarian Pizza - Acadian Pie',
              'Vegetarian Pizza - Bahn Mi',
              'Pizza - Roman Artichoke', # this might have meat but I see no indication
              'Italian Mezze', # mezze platters are usually vegetarian but who knows 
              'Pizza East Sider', # no idea but I guess this is just some kind of cheese pizza
              'Special- Whip Dip', # probably whip cream?]
              'Wholesome Noodles', # guessing
              'Summer Salad Special', # guessing 
              'Vanilla Apple Swiss Roll', # guessing 
              "Pizza - I'M A Pepper", # someone adds pepperoni to this which suggests default no meat
              'Special - Dessert', # most desserts are vegetarian not vegan
              'Bake-At-Home Cookies 4Pk',
              'Pizza - Black + White', # ricotta and olives? who knows
              'Pizza - Goodbye, Son', # ? 
              'Pizza - Mama',
              'Pizza - Mitchell Is 30!',
              'Pizza Mrs Robinson',
              'Lemon Puff',
              'Special Salad',
              'Pizza - I Dream Of Aubergine', # assuming eggplant pizza
              'Pizza Mediterrano', # guessing
              'Pizza - Autumn Pie',
              'Pizza - Spring Market',
              'Pizza Fun Guy Pie',
              'Special - Sasha‚Äôs Sticky Buns',
              'Cake Pop 4-Pack (Wdrwdr Limited Edition Flavor)',
              'Lemon Make You Happy Cake',
              'Pizza - The Ryan Gosling',
              'Special - Toast',
              'Special Beignets',
              'Pico De Greeko',
              'Pizza Budalk Fire',
              'Croustade',
              'Pizza Shakshuka',
              'Bagel Special', # totally guessing though it was ordered just once
              'Greek Mezze',
              'Pizza - Shaksuka Pie',
              'Pizza Asparagus',
              'Pizza Insalata',
              'Pizza Potato']

              
vegan = ['Vegan Veggie Sandwich',
         'Vegan Beyond Burger',
         'Vegan Apple Salad',
         'Side Slaw',
         'Bacon Side',
         'Vegan Impossible Burg',
         'Vegan Soba Salad',
         'Vegan Winter Grain Bowl',
         'Vegan Chard',
         'Vegan Summer Grain Bowl',
         'Vegan Harvest Salad',
         'Vegan Asparagus Salad',
         'Asparagus Salad with Chicken',
         'Vegan Grain Bowl',
         'Vegan Fried Cauliflower',
         'Vegan Fall Apple Salad',
         'Vegan Melon Salad',
         'Vegan Spring Salad',
         'Vegan Spring Market Salad',
         'Vegan Wheatberry Salad',
         'Vegan Summer Bean Salad',
         'Vegan Ratatouille Sandwich',
         'Vegan Berry Salad',
         'Vegan Berry Panzanella Salad',
         'Vegan Beet Salad',
         'Vegan Strawberry And Rhubarb Salad',
         'Vegan Special Cauliflower',
         'Calabrian',
         'Quarry',
         'Dd Beans',
         'Kids "A"B&J',
         'Side Of Sauce', 
         'Side Of House Hot', 
         'Hot Sauce - House Hot',
         'Hot Sauce - Chili Of The Corn',
         'Special- Cactus Fries',]



# If they are not vegetarian or vegan, they can default be labeled as animal-based, so the third list is unnecessary

# Supplemental labels, unnecessary for now

non_alcoholic_drinks = ['Coke', 'Diet Coke', 'Sprite', 'Iced Tea', 'Lemonade', 'Kids Milk',
                       'Coffee',  'Lacrooix', 'Cold Brew', 'Athletic', 'Barista 12', 'Water', 
                        'N/A', # N/A is non-alcoholic
                       
                       ]

alcoholic_drinks = ['Oz', 'Can', 'juice', 'Juice', 'Beverage', 'Crowler', 'Sour', 'IPA', 'Stout', 'Farmhaus', 'Pack',
                    'Sweet Potato', 'Razzy', 'Mango Tartshake', # all beers -- anything with a fruit or or milk name or pun is a beer
                    'Stf 16Oz', 'AtomWeight', 'Sweet Amarillo', 'Pineapple King', 'Dropping Smooth Beats',
                    'Passionfroeder 12Oz',  'Pineapple', 'Cbw Milk','Kiwis', 'Mango', 'Sg Pb Bananas',
                    'Sh Passion Fruit', 'Barley',  'Pineapple','Wr Strawberry Basil', 'Caramel Moochiato',
                    'Wr Medjool Date', # think it's a beer 
                    'Winter Blend16', # think it's a 16 oz cider
                    'Bookhouse Smooth Sailing', 'Sg CBW milk', 'Wr Cranberry',
                    'Wr Blackberry', 'North Coas Cran', 'Orange X', 'Mango Paletas', 'Sh Peach', 
                    'Sg', 'Downeast', 'Mt Dreamsicle 16', 'Raspberry Lime-Ade', 'S Fever', 'S Potion',
                    'Javahead', 'Orange', 'Sh Catchweight', 'Murder Ballads', 'Amber', 'Noble Brat',
                    'Embers Only', 'Mild', 'Os Black Widow Üï∑', 'Coup D‚Äôeclat', 'The Schnitz',
                    'Pepper Ann Sue', 'Slabtown', 'Rye Guy', 'You Here?', 'Spirit Animal',
                    'Cheetah', "Kilbane", # think it's part of kilbane's irish nitro stout
                    'Murder 12', 'Oktober', 'Gold', 'Samba No. 5', 'Stf', '666', 'Good Stuff',
                    'Crunchy', # crunch mod is 100z which I guess makes it a beer?
                    'Qwerty', # it's a blackberry beer https://www.facebook.com/noblebeastbrewingco/photos/a.1987008491362755/5193174207412818/?type=3&locale=ms_MY
                    'Subatomic Parachute', 'Azimuth', 'Velosette', 'Tfs', # team fortress? can't find it but mods include oz
                    'Studio C', 'Dearly Defarted', 'Hi, I Think You‚Äôre Great!','Baker Boy', 'Slab Town',
                    'Peacemaker', 'Peace Maker', 'Binary SoloÜ§Ñ', '3D', 'Grodz', 'Say When', 'Fever', 'Sonic',
                    'Dreamcoat', 'Grisette', 'Noblesibiling', 'The Baker', 'Russian', 'Wildblomen',
                    'Porter', 'Gpr 12', 'Styrian Woof', 'Bitter', 'Sh Stf', 'Widowmaker',
                    'Sh 666 Conducer', 'Sh Koth', 'Sh Heritage', 'Sh Dearly Departed',
                    'Sh Cookies', 'Sh Pppppp', 'Sh Say When', 'Sh Gem Cutter', 'Amber',
                    'Sh Ego', 'Sh Space', 'Sh Atomweight', 'Sh Binary Solo', 'Lobo Oscuro',
                    'Young Grizzlee', 'Sh 3Rd Times', 'Shacks', 'Sh Fall Farmhaus', 'Sh Chinchilla',
                    'Sh Crunchy', 'Ba Murders', # it's murder ballads, so I am wondering what Sh stands for
                    'Azimuth', 'Sh Elec', 'Eau Rouge', 'Sh War', 'Mb 2020', 'Sh Lichtenheiner', 'Pappy‚Äôs',
                     'Murder', 'Wr Oaked', 'Vermonter', 'Sh Studio C', 'Bakers Russian', 'Sg ', 'Sh ', 'Hog Heaven',
                    'C + M', 'Cnm',  # I think it's cookies and milk stout and a lot of variants
                    'Evil Twin', 'Hop Ryot', 'Witness Me', 'Mad Tree12', 'Young Grizz',
                    'G Evil Motives', 'Keller Door', 'Get Yer Oats', 'Bookhouse', 'After Dark',
                    'Dearly Depated', 'Bba Cookies 2020', 'Mystery Peaches', 'Double Evil', 'G Atomweight',
                    'G Dearly', 'G Studio C', 'G Ego', 'Ash‚Äôs 1St Day', 'G Embers Only', 'G Dearly Departed', 
                    'G Say When', 'G War Canary', 'G Drooping Swizz Beatz', 'Burley Man', # I think Burley Man is a cider
                    'Pickle Rick', 'Lil Buff Boys', 'Star Cut', # think Star Cut is from a Michigan brewery
                    'Pecan 16', # total guess based on name
                    'Wdrwdr', # it's a barleywine https://www.noblebeastbeer.com/product/wdr-barleywine-tasting/1105
                    '5 Rabbit', # 5 Rabbit Cerveceria I think
                    '12 dogs', # https://thirstydog.com/portfolio-item/12-dogs-of-christmas/
                    'Wr Cucumber', 'Wr Pumpkin', 'Avery', "The Baker'S Russian Cake Pop", 'Death Unicorn',
                    'Mikkellar', 'Red', 'Trampoline16', '3 Out Of 5', #no specific knowledge of Red, Trampoline, or 3 out of 5, just assuming a brewery sells beer
                    'Pizzasaurus Rex', # https://www.facebook.com/noblebeastbrewingco/posts/2617892278274370/?_rdr
                    '3Rd Times', '2017 C + M', 'Dudes Rug', 'Insetto', # https://www.facebook.com/groups/1648924892056363/posts/1907486406200209/
                    "Lil' B", # this one is weird with regex because it has a ' so I hope that switching from ' to " is ok. It's the lil' b beer from evil twin I think
                    'Third Wheel', 'French Oak', 'Matriarch', 'Old Choco12', 'Dead Rise', 'Bucket', # probably a bucket of beers
                    'La Roja', 'New Growth', 'Abtsolution10', 'Vida Y Muerte12', 'Yum Yum',
                    'Jesus', 'Lost Abbey', 'Goodbye Forver', 'Singlecut',  'Lion 12', 'The Cloud',
                    'Ba Oro', 'Eugene', 'Saucy16', 'Milkman12', 'Terrestrial Raw El Hanout 12Oz',
                    'The Madam', 'G Cookies', 'B Nektar', 'Afterburner', # https://www.instagram.com/beergoddesses/p/C8xJtqHukva/?locale=es_US&hl=en
                    '1Z Enuff',  'Gpr', 'Blackjack12', 'Whiner16', 'Roundhouse', 'Citrus',
                    'B Nectar', 'Bfod', 'Tuco 12', 'Wari', 'G 666', 'Hando', 'Bitches 12',
                    'Absolution', 'Aviator12', 'Fuego 10', 'Raspberry G', 'Burial10',
                    'G Heritage', 'S Saf', 'Lil B', 'S Dearly Departed', 'Pb Fosters',
                    'Aviator', 'Gabf Wdrwdr', 'Maine 16', 'White Monkey10' 'G Farm','G Hammers', 
                    'G Noble Sib', 'G Peacemaker', 'G Velocity Broker', 'Grod', 'S 666',
                    'S Say When', 'S-Eugene', 'G War',  'G Amarillo', 'G Azimuth', 
                    'G Hammer', 'G Okt', 'G Ol Bonzo', 'S Chrome', 'S Ego Tripping',
                    'S Farmhaus', 'S War', 'Sg-Eugene', 'Working Class', 'S Catchweight',
                    'Bitches12', 'S Moon Gold' 'S Stf', 'S Tfs', 'S Heritage', 'S Amber', 
                    'Sp Wari', 'G After Dark', 'G Amarillo', 'G Grodz', 'G Potato', 'Revolution 12',
                    'Cucumber Farm', 'Rosa',  'Yellow House Wooster Pike', 'Yellow House Pleasant Street', # these are guesses
                    'Danger Dust Do Not Sell', 'Sunshine', 'S Cookies', 'Comeback City',
                    'Evil 1/2', 'Foeder', # I wonder if this is the actual cask rather than a cask-aged beer, but no way to tell
                    'St White Monkey', 'Swintons Pride']

merch = ['Sticker', 'Koozie', 'Little Jacket', 'Blue And Red Shirt', 'Wine & Gold Shirt', 'Red T-Shirt',
         'Beanie', 'Red Beanie', 'Brown And Orange Shirt', 'Nbbc Shirt - Black', 'Noble Beast Shirt - Brown And Orange', 'Tank',
         'T-Shirt', 'Tshirt', 'T Shirt', 'Tee', 'Dad Hat!', # https://www.instagram.com/noble_beast_brewing/p/CP86bTtJaOI/?img_index=1
         'Navy And White', 'Hoodie', 'Brown T', 'Hat', 'Employee', 'Sleeve', 'Shipping - Usps Medium Flat Rate Box', 'Navy And Red',
         'Football Ringer T', '3/4 Raglan Gray', 'Cat‚Äôs Meow!!!',  # I think the cat's meow is a blockprint https://docs.google.com/document/d/1zyQGt7hMgV3iKj8vKlPMgS1ese2L3-ACGZixJAT0re0/edit?tab=t.0#heading=h.778fyscdi0hf
         'Retro Cavs', 'Red 3/4', # it's a '3/4 zip based on mods
         'Christmas', 'Crewneck', 'Ugly Christmas Sweater', 'Christmas Sweater', 
         'Sashajoe‚Äôs Treat Of The Day', 'Pocket T ', 'Beanie (No Pom)',
         'Ringer T', 'Flower Pin', '99 Wdrwdr22 Wax Tops', 'Button', 'Chrome', 
        'Classic', 'Emp Patch', 'Noble Beast - Red And Navy',]
rare = []

chicken = []

fish = ['Special- Lingood Chowder']

unknown = ['Soup', 'Vivanco', 'Rietos', 'Zagans', 'Katg', 'Special- Soup', 'Special Soup',
          'Special - Bird Soup', 'Special - Double Down Dip', 'Treat Of The Day',
          'Carina‚Äôs Treat Of The Day', 'Combo', 'Patch', 'Spiced',
          'Add Modifier', # Only 4 items but there's no easy way to clarify this
          'Saucy', 'Combo (W/ Widow Or Cookies)', 'Sauce', 'Hammers', 'Herb',
           'Lu', 'Nob Sib', 'Polka',] 

items_to_remove = ['Nye Party Ticket', 'Nye Designated Driver Ticket', 'Nb Gift Card', 
                   'Door Charge',
                   
                ] # nonfood

df_relabeled = fully_relabel_and_consolidate(
        df_uncleaned,
        remove=items_to_remove,
        modification_name_changes=modification_name_changes,
        vegan_list=vegan,
        vegetarian_list=vegetarian,
        meat_list=meat,
        drinks_list=non_alcoholic_drinks,
        alcohol_list=alcoholic_drinks,
        merch=merch,
        rare=rare,
        unknown=unknown,
        remove_categories=['Alcohol','Drink']
)

df_relabeled.to_parquet(f"data/4_palate_data_parquet_relabeled/relabeled/{loc_id}_sales_and_menu.parquet")

df_consolidated = df_relabeled.pipe(rename_items, name_changes = {'Beyond Burger': ['Vegan Beyond Burg','Vegan Beyond Burger','Beyond Burg', 'Beyond Burger','Beyond Burg with Bacon','Beyond Burger with Bacon'],
                                                                  'Impossible Burg' : ['Vegan Impossible Burg', 'Impossible Burg', 'Impossible Burg with Bacon']
                                                                  })

plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, top_n=30)

df_consolidated.to_parquet(f"data/4_palate_data_parquet_relabeled/consolidated/{loc_id}_sales_and_menu.parquet")

In [ ]:
df_consolidated.query('item_modifications.str.contains("Impossible") or item_name.str.contains("Impossible")')

In [ ]:
print(df_consolidated.query('item_modifications.str.contains("Vegan|Impossible|Beyond") or item_name.str.contains("Vegan|Impossible|Beyond")')[['item_name','item_modifications']].value_counts().to_string())

In [ ]:
# The five commands:

In [ ]:
### 1. All items:
print(df['item_name'].value_counts().to_string())


In [ ]:
### 2. Non merch, non drink, non alcoholic, items labeled as plant-based, or Unsure, along with category:
print(df.query('~dish_category.isin(["Merch","Drink","Alcohol"]) and is_plant_based == "Unsure"')
      .groupby(['item_name', 'dish_category']).size()
      .sort_values(ascending=False)
      .to_string())


In [ ]:
### 3. Modifications for a specific item:
print(df.query('item_name == "Special - Bird Soup"')['item_modifications'].value_counts().to_string())


In [ ]:
### 5. Check item category
df.query('item_name == "Wr Cucumber"')['dish_category']

In [ ]:
### Checking for beer 

# anything with 'oz' in it
print(df[df['item_name'].str.contains(r'oz', case=False)]['item_name'].value_counts().to_string())

In [ ]:
# Here's a custom thing to try to see which items in the Unsure category have an ounce number modification
# Get the list of unsure items
unsure_items = df.query('~dish_category.isin(["Merch","Drink","Alcohol"]) and is_plant_based == "Unsure"')['item_name'].unique()
print(f"Found {len(unsure_items)} unsure items")
print(unsure_items[:10])  # Print the first 10 as a sample




In [ ]:
# Check the data type and sample values in the item_modifications column
print(f"Data type of item_modifications: {df['item_modifications'].dtype}")
print("\nSample modifications:")
print(df['item_modifications'].dropna().sample(5).to_string())



In [ ]:
# Look for any items with "Oz" in modifications
oz_items = []
for item in unsure_items:
    mods = df[df['item_name'] == item]['item_modifications'].dropna()
    if not mods.empty:
        for mod in mods:
            if 'oz' in str(mod).lower() or 'OZ' in str(mod):
                oz_items.append((item, mod))
                break

print(f"Found {len(oz_items)} items with 'oz' in modifications")
# Print ALL items with oz modifications
for item, mod in oz_items:
    print(f"- {item}: {mod}")

